In [0]:
df = spark.read.table('data.ipldata.customer_shopping_behavior')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
df.printSchema()

In [0]:
# Count number of customers by Gender. 
gender_counts = df.groupBy('Gender').count()
display(gender_counts)

In [0]:
# Find total purchase amount by Category. 
categoty_sales = df.groupBy("Category").agg(f.round(f.sum("Purchase Amount (USD)"),2).alias('amount'))
display(categoty_sales)

In [0]:
# Get average purchase amount by Gender. 
gender_avg_amount = df.groupBy("Gender").agg(f.round(f.avg("Purchase Amount (USD)"),2).alias('amount'))
display(gender_avg_amount)

In [0]:
# Count customers by Subscription Status. 
sub_count = df.groupBy("Subscription Status").agg(f.count("Customer ID").alias('count'))
display(sub_count)

In [0]:
# Find top 5 locations with highest total spending. 
top_location = df.groupBy("Location").agg(f.round(f.sum("Purchase Amount (USD)"),2).alias("amount"))\
    .orderBy(f.desc("amount")).limit(5)
display(top_location)

In [0]:
# Get number of purchases by Season. 
season_purchase = df.groupBy('season').agg(f.count('Customer ID').alias('count'))
display(season_purchase)

In [0]:
# Find most commonly used Payment Method. 
common_method = df.groupBy('payment method').agg(f.count('Customer ID').alias('count'))\
    .orderBy(f.desc('count')).limit(1)
display(season_purchase)

In [0]:
# Get average Review Rating per category. 
category_avg_rating = df.groupBy('category').agg(f.round(f.avg('review rating'),2).alias('avg_rating'))
display(category_avg_rating )

In [0]:
# Get total revenue generated by each Shipping Type. 
shipType_rev = df.groupBy("shipping type").agg(f.round(f.sum("Purchase Amount (USD)"),2).alias('amount'))
display(shipType_rev)

In [0]:
# Count how many times discount was applied. 
discount_count = df.filter(df['discount applied'] == 'Yes').count()
display(discount_count)

In [0]:
# Rank customers based on total purchase amount. 
w = Window.orderBy(f.desc('amount'))
rank_cust = df.groupBy('Customer ID').agg(f.round(f.sum('Purchase Amount (USD)'),2).alias('amount'))\
    .withColumn('rnk',f.rank().over(w))\
        .select("Customer ID","amount","rnk")
display(rank_cust)

In [0]:
# Find top 3 customers per Location by spending. 
w = Window.partitionBy('Location').orderBy(f.desc('amount'))
top_location_cust = df.groupBy('Customer ID','Location').agg(f.round(f.sum('Purchase Amount (USD)'),2).alias('amount'))\
    .withColumn('rnk',f.rank().over(w))\
    .filter(f.col('rnk') <= 3)\
    .select('location',"Customer ID","amount","rnk")
display(top_location_cust)

In [0]:
# Calculate percentage contribution of each Category to total sales. 
total_amount = df.agg(f.sum("Purchase Amount (USD)").alias('total_amount')).collect()[0]['total_amount']
category_percent = df.groupBy("Category").agg(f.sum("Purchase Amount (USD)").alias('cat_amount'))\
    .withColumn('percent',f.round(f.col('cat_amount')*100/total_amount,2))\
    .select('Category','percent')
display(category_percent)

In [0]:
# Find customers whose purchase amount is above overall average. 
avg_amount = df.agg(f.avg("Purchase Amount (USD)").alias("avg_amount")).collect()[0]["avg_amount"]
above_avg_cust = df.filter(f.col("Purchase Amount (USD)") > avg_amount)\
    .select("Customer ID", "Purchase Amount (USD)")
display(above_avg_cust)

In [0]:
# Identify repeat customers (based on Previous Purchases). 
repeat_customers = df.filter(f.col('Previous Purchases') > 1).select('Customer ID', 'Previous Purchases')
display(repeat_customers)

In [0]:
# Find most popular item in each Category. 
w = Window.partitionBy('Category').orderBy(f.desc('count'))
pop_item = df.groupBy('Category','Item Purchased').agg(f.count('Customer ID').alias('count'))\
    .withColumn('rnk',f.rank().over(w))\
    .filter(f.col('rnk') == 1)\
    .select('Category','Item Purchased')
display(pop_item)  